# QPO训练演示

此notebook用于快速启动QPO训练和调试代码

In [1]:
# 导入模块

import os, re
import argparse
import torch
import datetime
import numpy as np
import random
import multiprocessing
import wandb
from agents import QPO
from envs import ToyEnv

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
# 定义参数
class SimpleArgs:
    def __init__(self):
        self.env_name = 'ToyEnv'
        self.algo_name = 'QPO'
        self.q_alpha = 0.25
        self.est_interval = 100
        self.log_interval = 100
        self.max_episode = 1000  # 改小以快速测试
        self.emb_dim = [8,8]
        self.init_std = np.sqrt(1e-1)
        self.gamma = 0.99
        self.seed = 0
        
        # 学习率参数
        self.theta_a = (10000**0.9) * 1e-3
        self.theta_b = 10000
        self.theta_c = 0.9
        self.q_a = (10000**0.6) * 1e-2
        self.q_b = 10000
        self.q_c = 0.6
        
        # 设备和路径
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")




In [3]:
# 固定随机种子
args = SimpleArgs()
random.seed(args.seed) 
np.random.seed(args.seed) 
torch.manual_seed(args.seed)

env = ToyEnv(n=10)
agent = QPO(args, env)

/opt/conda/envs/QPO/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
wandb: Currently logged in as: 1206052611 (1206052611-ecnu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Detected [agents] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


QPO warm up || n_epi:0500 0.25-quantile:-3.064


In [4]:
agent.actor(torch.tensor([1.0 for _ in range(0,10)]))

tensor([0.1158], grad_fn=<SqueezeBackward4>)

In [5]:
def indicator(x, y:torch.Tensor):
    """
    输入x一个标量或者张量, 输入y一个张量
    输出一个与y形状相同的张量, 当y<=x时, 输出1, 否则输出0
    """
    return torch.where(y <= x, torch.ones_like(y), torch.zeros_like(y))

In [6]:
y = torch.tensor([1,0,0,0,0,0,0,0,0,0])
x = 0.5

In [7]:
torch.where(y<=x, torch.tensor([i for i in range(0,10)]), torch.tensor([-1 for _ in range(0,10)]))

tensor([-1,  1,  2,  3,  4,  5,  6,  7,  8,  9])

In [8]:
[0]*10

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

In [9]:
a = torch.tensor([[1,2,3]])

In [10]:
a.reshape((3,))

tensor([1, 2, 3])

In [11]:
a.shape

torch.Size([1, 3])

In [12]:
state = env.reset()
state

array([0., 0., 0., 1., 0., 0., 0., 0., 0., 0.])

In [13]:
action = agent.choose_action(state)
action

array([0.60672134], dtype=float32)

In [14]:
agent.env.step(action)

(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]),
 -0.6728496744416732,
 False,
 {})

In [15]:
agent.actor(torch.from_numpy(state).float())

tensor([0.4515], grad_fn=<SqueezeBackward4>)

In [16]:
torch.diag(torch.exp(2*agent.actor.log_std))

tensor([[0.1000]])

In [17]:
from torch.distributions import MultivariateNormal
state = torch.from_numpy(state).float()

In [18]:
mean = agent.actor(state)
var = torch.diag(torch.exp(2*agent.actor.log_std))

In [19]:
dist = MultivariateNormal(mean, var)
action = dist.sample()

In [20]:
action_logprobs = dist.log_prob(action)

In [21]:
action_logprobs.shape

torch.Size([])

In [22]:
action.shape

torch.Size([1])

In [23]:
agent.memory.get_len()

0

In [24]:
a = [torch.tensor([1,2,3]),torch.tensor([3,4,5])]

In [25]:
torch.stack(a)

tensor([[1, 2, 3],
        [3, 4, 5]])

In [27]:
agent.compute_discounted_epi_reward()

(tensor([], device='cuda:0'), tensor([0.], device='cuda:0'))

In [ ]:
agent.update()

wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.
